# Citation Count Prediction Pipeline — 1993 PILOT RUN

This is a **single-year (1993) scoped-down version** of the full 1993–2003 pipeline,
meant to validate every stage end to end — data loading, LLM topic extraction,
trending-topic vocabulary, similarity scoring, baselines, and deep learning models —
on a small, fast subset (1,943 papers) before committing to a multi-hour run across
all 11 years (~30,351 papers).

Once this runs cleanly, the only change needed to scale up is `YEARS = list(range(1993, 2004))`
in the full pipeline notebook (`citation_prediction_pipeline_1993_2003.ipynb`) — the code
here is identical in structure, just scoped to one year.

**Data-quality fixes already applied (verified against your actual files):**
- Article IDs zero-padded to 7 digits before file lookup (needed for 2000+ papers; not
  relevant for 1993 itself, but kept here so this notebook stays a true drop-in template).
- Target year is **2022** (last year present in the citation-count file) — note the task
  description mentions "2023" but no 2023 column exists in the provided data.
- Trending-topic threshold: papers with more than 20 citations in 2022.

**What runs immediately (no LLM needed):** Sections 0, 1, 6.
**What needs your local Ollama:** Section 2.


In [25]:
import os, json, glob, time, re
import numpy as np
import pandas as pd
from pathlib import Path

# ---- EDIT THIS to point at your extracted dataset folder ----
DATA_ROOT = Path(r"C:\Users\vinipumba\Desktop\dataset")   # expects DATA_ROOT/"citation count", DATA_ROOT/"full text", DATA_ROOT/"deeptext"

CITATION_DIR = DATA_ROOT / "citation count"
FULLTEXT_DIR = DATA_ROOT / "full text/1993"
DEEPTEXT_DIR = DATA_ROOT / "deeptext/1993"

PILOT_YEAR = 1993                          # <-- the only thing scoped down vs the full pipeline
TARGET_YEAR = 2022                         # last year present in the data
FEATURE_YEARS = list(range(1993, 2022))   # integers, matching the actual column names  # 1993-2021 inclusive

np.random.seed(42)


In [30]:
import sys
print(sys.executable)

C:\Users\vinipumba\anaconda3\envs\citepred\python.exe


In [31]:
import sys
!{sys.executable} -m pip install torch --index-url https://download.pytorch.org/whl/cu121

'C:\Users\vinipumba\anaconda3\envs\citepred\python.exe' is not recognized as an internal or external command,
operable program or batch file.


## 0. Setup

## 1. Load the 1993 citation label file

Loads just `citation count/1993.xlsx`, and checks full-text / deep-text file coverage
for that year only.


In [26]:
def load_pilot_labels(citation_dir=CITATION_DIR, year=PILOT_YEAR):
    df = pd.read_excel(citation_dir / f"{year}.xlsx")
    df["pub_year"] = year

    year_cols = sorted([c for c in df.columns if isinstance(c, int)])
    for c in year_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[year_cols] = df[year_cols].fillna(0)

    # zero-pad Article Id to 7 digits so it matches the .txt filenames
    df["aid_str"] = df["Article Id"].astype(str).str.zfill(7)

    dup = df["Article Id"].duplicated().sum()
    if dup:
        print(f"Warning: {dup} duplicate Article Ids in {year}.xlsx")

    return df, year_cols

pilot_df, YEAR_COLS = load_pilot_labels()
print(f"{PILOT_YEAR} papers:", len(pilot_df))
print("Year columns:", YEAR_COLS[0], "-", YEAR_COLS[-1])
pilot_df.head(3)


1993 papers: 1943
Year columns: 1993 - 2022


,Article Id,Title,Author,Cited By,1993,1994,1995,1996,1997,1998,...,2015,2016,2017,2018,2019,2020,2021,2022,pub_year,aid_str
0,9301001,gonihedric string and asymptotic freedom,g.k.savvidy,75,1,5,2,5,8,4,...,1,0,4,2,0,1,1,0,1993,9301001
1,9301002,Scaling behavior of quantum four-geometries,"I Antoniadis, PO Mazur, E Mottola",67,2,5,6,4,3,5,...,2,2,0,0,0,0,2,2,1993,9301002
2,9301003,summing over inequivalent maps in the string t...,joseph a. minahan,68,7,12,12,4,3,4,...,1,2,0,0,0,0,1,2,1993,9301003


In [27]:
def check_text_coverage(df, base_dir, label, year=PILOT_YEAR):
    def has_text(row):
        p = base_dir / str(year) / f"{row['aid_str']}.txt"
        return p.exists()
    covered = df.apply(has_text, axis=1)
    print(f"{label} coverage for {year}: {covered.mean()*100:.1f}%  ({covered.sum()}/{len(df)} papers)")
    return covered

pilot_df["has_fulltext"] = check_text_coverage(pilot_df, FULLTEXT_DIR, "full-text")
pilot_df["has_deeptext"] = check_text_coverage(pilot_df, DEEPTEXT_DIR, "deep-text")


full-text coverage for 1993: 0.0%  (0/1943 papers)
deep-text coverage for 1993: 0.0%  (0/1943 papers)


## 2. LLM-based technical topic extraction (1993 papers only)

Runs a locally hosted Ollama model over each 1993 paper's text and asks for the top 5
**technical** topics. Checkpointed to a JSONL file so it's safe to interrupt/resume.
With ~1,900 papers per text type this should take well under an hour — use it to sanity
check prompt quality and output format before scaling to the full corpus.

Run this cell twice: once with `TEXT_DIR=FULLTEXT_DIR`, once with `TEXT_DIR=DEEPTEXT_DIR`.


In [28]:
import torch, json, re, time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none found")

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # requires HF login/token approval; swap for an open model if you don't have access
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map=DEVICE
)

TOPIC_PROMPT = '''You are a physics research assistant. Read the following excerpt from a
High-Energy Physics - Theory (HEP-TH) research paper and identify the TOP 5 TECHNICAL
topics it addresses. Topics must be specific, technical subfield terms (e.g. "conformal
field theory", "D-brane dynamics", "AdS/CFT correspondence") - NOT generic words like
"physics" or "theory". Return ONLY a JSON list of exactly 5 strings, nothing else.

PAPER TEXT:
{text}

JSON list of 5 technical topics:'''

MAX_CHARS = 8000

def extract_topics_gpu(text, retries=2):
    prompt = TOPIC_PROMPT.format(text=text[:MAX_CHARS])
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(DEVICE)
    for attempt in range(retries):
        try:
            with torch.no_grad():
                out = model.generate(inputs, max_new_tokens=150, temperature=0.1, do_sample=False)
            raw = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
            match = re.search(r"\[.*\]", raw, re.DOTALL)
            if match:
                topics = json.loads(match.group(0))
                topics = [str(t).strip() for t in topics][:5]
                if len(topics) == 5:
                    return topics
        except Exception as e:
            if attempt == retries - 1:
                print("failed:", e)
    return None

def run_topic_extraction_gpu(df, text_dir, out_path, text_col, year=PILOT_YEAR):
    out_path = Path(out_path)
    done = set()
    if out_path.exists():
        with open(out_path) as f:
            for line in f:
                try:
                    done.add(json.loads(line)["Article Id"])
                except Exception:
                    pass
        print(f"Resuming: {len(done)} papers already processed.")

    subset = df[df[text_col]]
    n_total = len(subset)
    n_done_now = 0
    t0 = time.time()
    with open(out_path, "a") as out:
        for i, row in subset.iterrows():
            if row["Article Id"] in done:
                continue
            p = text_dir / str(year) / f"{row['aid_str']}.txt"
            try:
                text = p.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            topics = extract_topics_gpu(text)
            if topics is None:
                continue
            out.write(json.dumps({"Article Id": row["Article Id"], "pub_year": year, "topics": topics}) + "\n")
            out.flush()
            n_done_now += 1
            if n_done_now % 50 == 0:
                rate = n_done_now / (time.time() - t0)
                print(f"  {n_done_now}/{n_total} processed ({rate:.2f} papers/sec)...")
    print("Done. Output at", out_path)

run_topic_extraction_gpu(pilot_df, FULLTEXT_DIR, "fulltext_topics_1993.jsonl", text_col="has_fulltext")
run_topic_extraction_gpu(pilot_df, DEEPTEXT_DIR, "deeptext_topics_1993.jsonl", text_col="has_deeptext")
# Uncomment to actually run (requires Ollama running locally):
# run_topic_extraction(pilot_df, FULLTEXT_DIR, "fulltext_topics_1993.jsonl", text_col="has_fulltext")
# run_topic_extraction(pilot_df, DEEPTEXT_DIR, "deeptext_topics_1993.jsonl", text_col="has_deeptext")


ModuleNotFoundError: No module named 'torch'

## 3. Trending topic list (from the full multi-year citation data)

Note: "trending" is defined by citations **in 2022**, which requires the full citation
history column even though the paper *set* here is scoped to 1993 — the 2022 citation
column is already present in `1993.xlsx` (it tracks each 1993 paper's citations through
2022), so no extra file is needed for this pilot.


In [18]:
def load_topics(jsonl_path):
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

def build_trending_vocab(topics_df, labels_df, target_year=TARGET_YEAR, min_citations=20):
    merged = topics_df.merge(labels_df[["Article Id", target_year]], on="Article Id", how="left")
    trending_papers = merged[merged[target_year] > min_citations]
    vocab = set()
    for t_list in trending_papers["topics"]:
        vocab.update([t.lower().strip() for t in t_list])
    print(f"Trending papers (>{min_citations} citations in {target_year}): {len(trending_papers)} of {len(topics_df)}")
    print(f"Trending vocabulary size: {len(vocab)}")
    return vocab

# fulltext_topics = load_topics("fulltext_topics_1993.jsonl")
# trending_vocab_fulltext = build_trending_vocab(fulltext_topics, pilot_df)
# deeptext_topics = load_topics("deeptext_topics_1993.jsonl")
# trending_vocab_deeptext = build_trending_vocab(deeptext_topics, pilot_df)


## 4. Percentage similarity

Percentage of each paper's 5 extracted topics that appear in the trending vocabulary.


In [19]:
def compute_similarity_pct(topics_df, trending_vocab):
    def pct(topics):
        topics_l = [t.lower().strip() for t in topics]
        overlap = sum(1 for t in topics_l if t in trending_vocab)
        return 100.0 * overlap / len(topics_l) if topics_l else 0.0
    out = topics_df.copy()
    out["similarity_pct"] = out["topics"].apply(pct)
    return out[["Article Id", "similarity_pct"]]

# fulltext_sim = compute_similarity_pct(fulltext_topics, trending_vocab_fulltext)
# deeptext_sim = compute_similarity_pct(deeptext_topics, trending_vocab_deeptext)


## 5. Feature assembly

Citation history (1993–2021) + similarity score → target citations in 2022, for the
1993 paper cohort only.


In [20]:
def assemble_features(labels_df, similarity_df, feature_years=FEATURE_YEARS, target_year=TARGET_YEAR):
    df = labels_df.merge(similarity_df, on="Article Id", how="inner")
    X_hist = df[feature_years].values.astype(float)
    X_sim = df[["similarity_pct"]].values.astype(float)
    X = np.hstack([X_hist, X_sim])
    y = df[target_year].values.astype(float)
    return df, X, y

# df_full, X_full, y_full = assemble_features(pilot_df, fulltext_sim)
# df_deep, X_deep, y_deep = assemble_features(pilot_df, deeptext_sim)


## 6. Baselines — run this FIRST, before any deep learning model

Only needs citation history (no LLM/similarity required), so it runs immediately on the
1993 data as-is. Compare every deep learning result in Section 8 against these numbers.


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    acc5 = (np.abs(y_true - y_pred) <= 5).mean() * 100
    acc10 = (np.abs(y_true - y_pred) <= 10).mean() * 100
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Acc5": acc5, "Acc10": acc10}

def run_baselines(labels_df, feature_years=FEATURE_YEARS, target_year=TARGET_YEAR):
    y = labels_df[target_year].values.astype(float)
    last_known = labels_df[feature_years[-1]].values.astype(float)  # citations_2021
    X_hist = labels_df[feature_years].values.astype(float)

    results = {}
    results["naive_persistence"] = eval_metrics(y, last_known)

    n_splits = min(5, len(labels_df))  # guard against tiny pilot subsets
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    preds = np.zeros(len(y))
    for tr, te in kf.split(X_hist):
        lr = LinearRegression().fit(X_hist[tr], y[tr])
        preds[te] = lr.predict(X_hist[te])
    results["linreg_full_history"] = eval_metrics(y, preds)

    preds2 = np.zeros(len(y))
    X1 = last_known.reshape(-1, 1)
    for tr, te in kf.split(X1):
        lr = LinearRegression().fit(X1[tr], y[tr])
        preds2[te] = lr.predict(X1[te])
    results["linreg_last_year_only"] = eval_metrics(y, preds2)

    return pd.DataFrame(results).T

pilot_baseline_results = run_baselines(pilot_df)
print(pilot_baseline_results.round(4))


                          MAE    RMSE      R2     Acc5    Acc10
naive_persistence      0.8775  2.3415  0.9352  96.9120  99.2795
linreg_full_history    1.0810  2.8474  0.9042  95.9341  99.1251
linreg_last_year_only  0.9695  2.4073  0.9316  96.0885  99.1251


## 7. Feature ablation: does similarity add value? (1993 pilot)

Once Section 5 has produced `X_full`/`y_full` (and/or `X_deep`/`y_deep`), run this to
test whether adding the similarity feature improves on citation history alone, even
at this small pilot scale.


In [22]:
def feature_ablation(X_hist_only, X_with_sim, y):
    n_splits = min(5, len(y))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    def cv_metrics(X):
        preds = np.zeros(len(y))
        for tr, te in kf.split(X):
            lr = LinearRegression().fit(X[tr], y[tr])
            preds[te] = lr.predict(X[te])
        return eval_metrics(y, preds)
    return {
        "history_only": cv_metrics(X_hist_only),
        "history_plus_similarity": cv_metrics(X_with_sim),
    }

# X_hist_only = X_full[:, :-1]   # drop the similarity column
# ablation = feature_ablation(X_hist_only, X_full, y_full)
# pd.DataFrame(ablation).T


## 8. Deep learning models: ANN, CNN, LSTM, MLP, RNN (1993 pilot)

Note: with only ~1,900 papers, held-out test sets here will be small (~380 with an
80/20 split) — treat these as a smoke test of the code, not as trustworthy accuracy
numbers. Report final numbers from the full 1993–2003 run, not this pilot.


In [23]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_ann(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

def build_mlp(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

def build_cnn(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.Conv1D(16, 3, activation="relu", padding="same"),
        layers.Conv1D(32, 3, activation="relu", padding="same"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return m

def build_lstm(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.LSTM(32, return_sequences=False),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

def build_rnn(input_dim):
    m = models.Sequential([
        layers.Input(shape=(input_dim, 1)),
        layers.SimpleRNN(32, return_sequences=False),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return m

MODEL_BUILDERS = {"ANN": build_ann, "MLP": build_mlp, "CNN": build_cnn, "LSTM": build_lstm, "RNN": build_rnn}
SEQUENCE_MODELS = {"CNN", "LSTM", "RNN"}

def train_and_eval_all(X, y, test_size=0.2, epochs=50, batch_size=32, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    results = {}
    for name, builder in MODEL_BUILDERS.items():
        model = builder(X.shape[1])
        Xtr = X_train[..., None] if name in SEQUENCE_MODELS else X_train
        Xte = X_test[..., None] if name in SEQUENCE_MODELS else X_test
        model.fit(Xtr, y_train, epochs=epochs, batch_size=batch_size, verbose=0,
                  validation_split=0.1,
                  callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
        preds = model.predict(Xte, verbose=0).flatten()
        results[name] = eval_metrics(y_test, preds)
        print(name, results[name])
    return pd.DataFrame(results).T

# dl_results_fulltext_1993 = train_and_eval_all(X_full, y_full)
# dl_results_fulltext_1993


## 9. Pilot summary

Once Sections 2–8 have been run for both full-text and deep-text, this compiles a
single comparison table so you can sanity-check the whole chain before scaling up.


In [24]:
def build_pilot_report(baseline_results, dl_results, config_name):
    combined = pd.concat([baseline_results, dl_results])
    combined.insert(0, "configuration", config_name)
    return combined

# pilot_full_report = build_pilot_report(pilot_baseline_results, dl_results_fulltext_1993, "FullText_1993_pilot")
# pilot_full_report.to_csv("pilot_results_1993.csv")
# pilot_full_report


### Next step

Once every cell above runs cleanly and the numbers look sane (in particular: compare
your deep learning R² against the naive persistence baseline in Section 6 — don't
report a deep learning result as a win unless it beats that number), switch to
`citation_prediction_pipeline_1993_2003.ipynb` and change nothing except the year range
(`YEARS = list(range(1993, 2004))`) to run the same pipeline on the full corpus.
